In [ ]:
from utils.constants import SEC_TO_INEV, KPC_TO_INEV, GCM3_TO_EV4
from utils.data_utils import read_medium_data, interpolate_data
from inputs.spectrum_sources import CSVSpectrum
from inputs.configs import PhysicsConfig, PropagationConfig
from utils.logging_utils import setup_logging, get_logger
from waveform.collection import WaveformCollection
from waveform.propagation import plot_spectrogram, export_source_parameters, create_source_from_propagation

# Enable inline plotting
%matplotlib inline

# Configure logging
setup_logging(log_file='propagation.log', level='INFO')
logger = get_logger()

# Complete Workflow: Spectrum → Propagation → Constraints

This notebook demonstrates the **complete pipeline** from input spectrum to constraint plots.

## What you'll get:
1. **Spectrogram visualization** (PDF) - Shows frequency vs. time evolution
2. **Constraint plot** (PNG) - Shows parameter space constraints
3. **Exported parameters** (optional) - Save for later use

## Workflow:
1. Load spectrum from CSV (bosenova collapse)
2. Propagate through galactic density profile
3. Generate spectrogram
4. Bridge to constraint plotting
5. Generate constraint plot

---

**Note:** For quick constraint plots without propagation, see `examples_constraint_plots_manual.ipynb`

In [ ]:
from utils.constants import YEAR_TO_SEC, DAY_TO_SEC

# --- User-defined Parameters ---

# Particle physics properties
mass = 1e-19                       # Scalar field mass [eV]
coupling = 1e22                    # Dilatonic coupling strength
K = 1e-3                           # Energy density fraction

# Propagation resolution parameters
density_num_points = 1000          # Density profile resolution
N_points_spectrogram = 1000        # Spectrogram time steps

In [ ]:
# Configure your source (Example: #TODO - describe this example explicitly)
amplitude_normalization = 0.3 / 1e-85  # Amplitude scaling for bosenova spectrum
burst_duration_factor = 400        # Burst duration: factor / (mass * SEC_TO_INEV)
burst_duration = burst_duration_factor / (mass * SEC_TO_INEV)

# Load bosenova spectrum from CSV with scaling applied directly
spectrum = CSVSpectrum(
    'data/BosonStarSpectrumRelOnly.txt',
    i_p=0, 
    i_A=1, 
    skip_header=False, 
    num_points=1000,
    scaled_momentum=mass,
    scaled_amplitude=lambda A: A * amplitude_normalization
)

In [ ]:
# Configure physics and propagation
physics = PhysicsConfig(mass=mass, coupling=coupling, K=K, burst_duration=burst_duration)
propagation_config = PropagationConfig('data/Source@R=1parsec.csv', density_num_points=density_num_points)
collection = WaveformCollection(spectrum, physics, propagation_config)

# Propagate
results = collection.propagate_all(N_points_spectrogram=N_points_spectrogram, save_waveform=False)

# Plot spectrogram
avg_density, arrival_window = plot_spectrogram(
    N_points_spectrogram, results['t_min'], results['t_max'],
    results['E'], results['spectrogram'],
    filename='plots/spectrogram_bosenova.pdf',
    show=True
)

In [ ]:
# Particle physics coupling labels (used for constraint plots)
coupling_type = 'electron'         # Coupling type: 'photon', 'electron', 'gluon'
coupling_order = 'quad'            # Coupling order: 'linear' or 'quad'
param_filename = 'bosenova.param'

# Export source parameters with coupling information
R = collection.density_profile[0][-1] - collection.density_profile[0][0] # Get distance from density profile
export_source_parameters(
    avg_density, burst_duration, R, mass,
    arrival_window=arrival_window,
    coupling=coupling,
    K=K,
    coupling_type=coupling_type,
    coupling_order=coupling_order,
    to_file=True,
    filename=param_filename
)

print(f"\nExported parameters to {param_filename}")

In [ ]:
from inputs.source import Source
from inputs.spectrum import SignalModel
from inputs.experiment import Experiment
from plotting.output_handler import OutputHandler
from plotting.plots import Plot
from utils.constants import SOLAR_TO_EV

# Create Source from propagation results (direct bridge - no file needed)
source = create_source_from_propagation(
    avg_density, 
    burst_duration, 
    R, 
    mass,
    arrival_window,
    coupling_type=coupling_type,
    coupling_order=coupling_order,
    ULB_type='scalar'
)

print('Created Source from propagation results:')
print(f'  mass = {source.mass} eV')
print(f'  R = {source.R} pc')
print(f'  tstar = {source.tstar} s')
print(f'  Etot = {source.Etot / SOLAR_TO_EV:.4f} solar masses')
print(f'  coupling_type = {source.coupling_type}')
print(f'  coupling_order = {source.coupling_order}')

In [ ]:
# Configure experimental parameters
integration_time = DAY_TO_SEC      # Integration time for transient search [s]
integration_time_DM = YEAR_TO_SEC  # Integration time for DM search [s]
sensitivity = 1e-17    

experiment = Experiment(
    integration_time=integration_time,
    integration_time_DM=integration_time_DM,
    sensitivity=sensitivity,
    time_delays={'day': DAY_TO_SEC, 'year': YEAR_TO_SEC}
)

# Create signal model
signal_model = SignalModel(source=source, experiment=experiment)

print('\nSignal model created - ready for constraint plotting')

In [ ]:
# Generate constraint plot
xlims = (1e-21, 6e-6)   # Mass axis range [eV]
ylims = (5e-21, 1e36)   # Coupling axis range

plot = Plot(xlims, ylims, exclude_mass=False)
output = OutputHandler(plot)
output.plot_parameter_space(source, signal_model, plot, save_path='plots/constraints_bosenova.png')

print('Constraint plot saved to plots/constraints_bosenova.png')

print('\n=== Workflow Complete ===')
print('Generated outputs:')
print('  1. plots/spectrogram_bosenova.pdf (from propagation)')
print('  2. plots/constraints_bosenova.png (constraint plot)')
print(f'  3. {param_filename} (exported parameters - optional for later use)')